In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class ContinuousSignal:
    def __init__(self,func):        
        self.func=func

    def shift(self, k):
        temp=self.signal
        newINF=self.INF+abs(k)
        newSignal=ContinuousSignal(newINF)

        #oldSignal er -INF => newSignal er startIdx
        startIdx=newINF-self.INF + k
        #oldSignal er +INF => newSignal er endIdx
        endIdx=startIdx+self.n
        newSignal.signal[startIdx:endIdx]=self.signal[:]
        return newSignal
            

    def add(self, other):
        inf=max(self.INF,other.INF)
        temp=ContinuousSignal(inf)
        
        #adding self
        start=inf-self.INF
        end=start+self.n
        temp.signal[start:end]=self.signal[:]

        #adding other
        start=inf-other.INF
        end=start+other.n
        temp.signal[start:end]+=other.signal[:]
        
        return temp


    def multiply(self, scalar):
        temp=ContinuousSignal(self.INF)
        temp.signal=self.signal*scalar
        return temp

    def plot(self, title="Discrete Signal"):
        t=np.arange(-self.INF,self.INF+1)
        
        plt.figure(figsize=(10,4))
        plt.stem(t,self.signal,basefmt=" ", linefmt='b-',markerfmt='bo')
        plt.title(title)
        plt.xlabel("n (time)")
        plt.ylabel("Amplitude")

        plt.grid(True,which="both")
        

class LTI_System:
    def __init__(self, impulse_response: ContinuousSignal):
        self.h=impulse_response

    def linear_combination_of_impulses(self, input_signal: ContinuousSignal):
        # Decompose the signal into impulses and corresponding coefficients
        impulses=[]
        coeff=[]

        for i,val in enumerate(input_signal.signal):
            if(val!=0):
                time=i-input_signal.INF

                pulse=ContinuousSignal(0)
                pulse.set_value_at_time(0,1)
                pulse=pulse.shift(time)

                impulses.append(pulse)
                coeff.append(val)
        return impulses,coeff

    def output(self, input_signal: ContinuousSignal):
        y=ContinuousSignal(0)

        impulses,coeff=self.linear_combination_of_impulses(input_signal)
        
        for delta, coeff in zip(impulses,coeff):
            i=np.argmax(delta.signal)
            time=i-delta.INF

            h_shifted=self.h.shift(time)
            h_scaled=h_shifted.multiply(coeff)

            y=y.add(h_scaled)

        return y